*The executable cells in this notebook run quick tests of the study workflow, not the full published experiment. The completed results appear in the final section, the [README](README.md) and [`results/`](results/). The original research workflow is preserved at [`study-v1`](https://github.com/MoritzDue/bermudan-neural-stopping/tree/study-v1).*

# 03 — Neural stopping and Longstaff–Schwartz

The main experiment compares exercise policies for a five-asset Bermudan max-call under risk-neutral GBM.

| Component | Completed experiment |
|---|---|
| Methods | Validation-tuned LSM and neural stopping with sorted asset inputs plus payoff |
| Training | 1,000, 10,000 and 50,000 paths; ten independent training seeds per budget |
| Validation | 5,000 paths per seed, separate from training and testing |
| Testing | One shared set of 200,000 paths |
| Neural settings | 1,500 Adam steps, learning rate 0.001, batch size 512 |
| Optimality check | One 50k, seed-zero policy per method, evaluated separately with nested simulation |

The 1k training budget still uses an additional 5k validation paths; it is not a 1k total simulation budget. A broader representation ablation, neural parameter search and the 2D comparison were not completed.

Sections 1–3 below are **quick workflow examples**. Section 4 records the actual completed study. Do not use quick-mode outputs as additional evidence for the reported conclusions.

In [ ]:
MODE = "quick"

if MODE != "quick":
    raise RuntimeError("Full results come from the frozen manifest runner at git tag study-v1; "
                       "this notebook runs quick smoke checks only.")

import json, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from stopping import *

print("Thread limits:", limit_threads(1))
BASE = CONFIG[MODE]["base_seed"]
OUT = results_dir(MODE)
print("Mode:", MODE, "| results ->", OUT)
INCLUDE_10D = False

study = CONFIG[MODE]["study"]
frozen_path = OUT / "frozen_settings.json"
frozen = load_study_settings(frozen_path, MODE)
settings = NeuralSettings(**frozen["neural"])
print("Frozen neural settings:", settings)
print("Study design:", json.dumps(study, indent=1))

selection_variants = frozen.get("selection_variants", CONFIG[MODE]["calibration"]["variants"])
blocking = {c: g for c, g in frozen.get("not_saturated_cells", {}).items() if c.split("@")[0] in selection_variants}
if blocking and MODE == "full":
    raise RuntimeError(
        f"Calibration: selection variants not saturated: {blocking}. "
        "Extend calibration within the approved compute budget before viewing main results, or pause with evidence.")
BUDGET_LABEL = "restricted optimisation budget" if blocking else "saturated optimisation budget (per calibration)"
if blocking:
    print(f"Calibration gate: {list(blocking)} not saturated -> results are labelled '{BUDGET_LABEL}'"
          + (" (quick mode: gate not enforced)" if MODE == "quick" else ""))
diagnostic_flags = [c for c in frozen.get("not_saturated_cells", {}) if c not in blocking]
if diagnostic_flags:
    print("Diagnostic variant still improving (Claim B evidence, report it):", diagnostic_flags)

## 1. Quick experiment

Fit and compare the methods on small simulated samples. The code saves completed runs so that a quick experiment can be resumed. This section is an implementation check, not the source of the full-study tables.

In [ ]:
cases = {5: paper_max_call(5), 2: paper_max_call(2)}
if INCLUDE_10D:
    cases[10] = paper_max_call(10)
plan = [(5, b, s, list(VARIANTS) if b in study["ablation_budgets"] else [MAIN_VARIANT])
        for b in study["budgets"] for s in study["seeds"]]
plan += [(2, study["case_2d"]["budget"], s, [MAIN_VARIANT]) for s in study["case_2d"]["seeds"]]
if INCLUDE_10D:
    plan += [(10, study["case_10d"]["budget"], s, [MAIN_VARIANT]) for s in study["case_10d"]["seeds"]]

run_info = dict(frozen=dict(mode=MODE, study=study, neural=frozen["neural"], lsm_grid=LSM_GRID,
                            base_seed=BASE, include_10d=INCLUDE_10D, variants=VARIANTS,
                            contracts={d: asdict(o) for d, o in cases.items()}, code_hash=code_hash(),
                            budget_label=BUDGET_LABEL, settings_source=frozen.get("settings_source", "matching calibration"),
                            blocking_cells=sorted(blocking)),
                environment=dict(threads=thread_report(), numpy=np.__version__, torch=torch.__version__),
                development_cost_cpu_minutes=frozen.get("development_cost_cpu_minutes"))
store = ResultStore(OUT / "study", run_info, MODE)
pre = OUT / "preflight.json"
if pre.exists() and json.loads(pre.read_text()).get("neural") == frozen["neural"]:
    print(f"Preflight projection: about {json.loads(pre.read_text())['projected_minutes']:.1f} minutes for the 5D + 2D units.")
print(f"{len(plan)} units planned, {len(store.done())} already finished.")

In [ ]:
tests, refs = {}, {}
for d, o in cases.items():
    refs[d] = reference_for(o)
    tests[d] = gbm_paths(o, study["test"], seed_for(BASE, "test", d))

t_start = time.perf_counter()
done = store.done()
todo = [u for u in plan if (cases[u[0]].name, u[1], u[2]) not in done]
for i, (d, budget, seed, variants) in enumerate(todo, 1):
    t0 = time.perf_counter()
    store.append(run_unit(cases[d], budget, seed, variants, settings, study["val"], tests[d], refs[d], BASE),
                 key=(cases[d].name, budget, seed))
    elapsed = time.perf_counter() - t_start
    print(f"[{i}/{len(todo)}] {d}D budget={budget:>6} seed={seed}  ({time.perf_counter() - t0:.0f}s; "
          f"elapsed {elapsed / 60:.1f} min, remaining ~{elapsed / i * (len(todo) - i) / 60:.1f} min)", flush=True)
print("All planned units finished.")

## 2. Policy comparison

LSM and neural stopping are evaluated on the same test paths within each training seed, so their payoff difference is paired. Confidence intervals across seeds describe variation in fitted policies **conditional on the shared test set**; they do not eliminate uncertainty from that test sample. The main results use policies selected with the validation guard.

In [ ]:
runs, pairs, dates = store.load("runs"), store.load("paired"), store.load("dates")
for name, table in (("runs", runs), ("paired", pairs), ("dates", dates)):
    table.to_csv(store.folder / f"{name}.csv", index=False)

def across_seeds(df, by, col):
    rows = []
    for key, sub in df.groupby(by):
        m, se, lo, hi = mean_ci(sub[col])
        rows.append(dict(zip(by, key if isinstance(key, tuple) else (key,)), mean=m, ci_low=lo, ci_high=hi, seeds=len(sub)))
    return pd.DataFrame(rows)

main = runs[(runs.d == 5) & runs.guarded & runs.method.isin(["lsm", f"nn_{MAIN_VARIANT}"])]
summary = across_seeds(main, ["budget", "method"], "gap").rename(columns={"mean": "gap"})
extra = main.groupby(["budget", "method"]).agg(value=("value", "mean"), median_test_se=("test_se", "median"),
                                                train_s=("train_s", "mean"), selection_s=("selection_s", "mean"),
                                                guarded_dates=("n_guarded_dates", "mean")).reset_index()
summary = summary.merge(extra, on=["budget", "method"])
uncertainty_rows = []
for (budget, method), sub in main.groupby(["budget", "method"]):
    uncertainty_rows.append(dict(budget=budget, method=method,
        **absolute_gap_uncertainty(sub.gap, sub.test_se)))
summary = summary.merge(pd.DataFrame(uncertainty_rows), on=["budget", "method"])
summary["optimisation_budget"] = BUDGET_LABEL
summary.to_csv(OUT / "summary_main.csv", index=False)
print("Gap = 26.159 - value (positive = below the reference). Reference uncertainty about ±0.02.")
summary.round(4)

### 2.1 Neural stopping minus LSM

In [ ]:
contrast = pairs[(pairs.d == 5) & pairs.guarded & (pairs.contrast == f"nn_{MAIN_VARIANT} - lsm")]
diff = across_seeds(contrast, ["budget"], "diff").merge(
    contrast.groupby("budget").paired_se.median().rename("median_paired_se").reset_index(), on="budget")
diff.to_csv(OUT / "summary_nn_minus_lsm.csv", index=False)
diff.round(4)

### 2.2 Illustrative plots

The plots generated here describe the quick-mode runs. They are separate from the two figures in the README, which are regenerated from committed full-study summaries using `make_figures.py`.

In [ ]:
ref5 = PAPER_REFERENCES[5]
band = (ref5["value"] - ref5["high"], ref5["value"] - ref5["low"])
budgets = sorted(summary.budget.unique())
labels = {"lsm": "tuned LSM", f"nn_{MAIN_VARIANT}": f"neural ({MAIN_VARIANT})"}
colors = {"lsm": "tab:blue", f"nn_{MAIN_VARIANT}": "tab:orange"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for method, sub in summary.groupby("method"):
    sub = sub.sort_values("budget")
    err = np.vstack([sub.gap - sub.combined_ci_low, sub.combined_ci_high - sub.gap]) if sub.combined_ci_low.notna().all() else None
    c = colors[method]
    axes[0].errorbar(sub.budget, sub.gap, yerr=err, fmt="o-", capsize=3, color=c, label=labels[method])
    axes[1].errorbar(sub.selection_s, sub.gap, yerr=err, fmt="o-", capsize=3, color=c, label=labels[method] + " (total cost)")
    axes[1].plot(sub.train_s, sub.gap, "x--", color=c, alpha=0.6, label=labels[method] + " (final fit only)")
for ax in axes:
    ax.axhspan(*band, color="grey", alpha=0.25, label="reference uncertainty")
    ax.axhline(0, color="black", lw=0.8); ax.set_xscale("log"); ax.set_ylabel("Gap to reference (26.159 - value)")
    ax.legend(fontsize=8)
axes[0].set_xticks(budgets, [f"{b:,}\n({b + study['val']:,})" for b in budgets])
axes[0].minorticks_off()
axes[0].set(xlabel="Training paths per seed\n(training + validation paths per seed)", title="Data efficiency (5D)")
dev = frozen.get("development_elapsed_minutes", frozen.get("development_cost_cpu_minutes", float("nan")))
axes[1].set(xlabel="CPU seconds per policy (single thread)",
            title=f"Compute efficiency (5D); one-off neural calibration: {dev:.1f} min elapsed")
fig.suptitle(f"Neural policy trained under a {BUDGET_LABEL}", fontsize=10)
fig.tight_layout(); fig.savefig(OUT / "fig_gap_vs_data_and_compute.png", dpi=160); plt.show()

### 2.3 Input-feature comparison

The code compares neural representations with and without sorting and an explicit payoff feature. This is an exploratory quick-mode check; the planned full-study ablation was not completed, so it cannot establish a representation advantage.

In [ ]:
abl_runs = runs[(runs.d == 5) & runs.guarded & runs.budget.isin(study["ablation_budgets"]) & runs.method.str.startswith("nn_")]
cells = across_seeds(abl_runs, ["budget", "method"], "value")
abl_pairs = pairs[(pairs.d == 5) & pairs.guarded & pairs.contrast.isin(list(ABLATION_CONTRASTS))]
effects = across_seeds(abl_pairs, ["budget", "contrast"], "diff").merge(
    abl_pairs.groupby(["budget", "contrast"]).paired_se.median().rename("median_paired_se").reset_index(),
    on=["budget", "contrast"])
cells.to_csv(OUT / "summary_ablation_cells.csv", index=False)
effects.to_csv(OUT / "summary_ablation_effects.csv", index=False)

fig, axes = plt.subplots(1, len(study["ablation_budgets"]), figsize=(5.5 * len(study["ablation_budgets"]), 3.8), squeeze=False)
for ax, b in zip(axes[0], study["ablation_budgets"]):
    c = cells[cells.budget == b].set_index("method")
    for sort_label in ("sorted", "raw"):
        names = [f"nn_{sort_label}", f"nn_{sort_label}+payoff"]
        if all(n in c.index for n in names):
            m = c.loc[names, "mean"].to_numpy()
            err = np.vstack([m - c.loc[names, "ci_low"], c.loc[names, "ci_high"] - m]) if c.loc[names, "ci_low"].notna().all() else None
            ax.errorbar([0, 1], m, yerr=err, fmt="o-", capsize=3, label=f"{sort_label} inputs")
    ax.axhspan(ref5["low"], ref5["high"], color="grey", alpha=0.25, label="reference")
    ax.set_xticks([0, 1], ["no payoff input", "payoff input"])
    ax.set(title=f"5D, {b:,} training paths", ylabel="Policy value (guarded)"); ax.legend(fontsize=8)
fig.suptitle(f"All variants trained under the same {BUDGET_LABEL}", fontsize=10)
fig.tight_layout(); fig.savefig(OUT / "fig_ablation.png", dpi=160); plt.show()
display(cells.round(4)); effects.round(4)

### 2.4 Validation guard

Compare policies with and without the rule that disables exercise on dates showing negative validation benefit. Both evaluations use the same fitted models; this is not an independently trained alternative policy.

In [ ]:
guard_pairs = pairs[pairs.contrast.str.endswith("guarded - unguarded")].copy()
guard_pairs["method"] = guard_pairs.contrast.str.split(":").str[0]
guard = across_seeds(guard_pairs, ["d", "budget", "method"], "diff").merge(
    runs[runs.guarded].groupby(["d", "budget", "method"]).n_guarded_dates.mean().rename("mean_guarded_dates").reset_index(),
    on=["d", "budget", "method"])
guard.to_csv(OUT / "summary_guard.csv", index=False)
by_date = dates.pivot_table(index=["d", "budget", "method"], columns="date", values="guard", aggfunc="mean")
display(guard.round(4))
print("Share of seeds in which the guard fired, by exercise date:")
by_date.round(2)

### 2.5 Other asset dimensions

The code can examine smaller or larger asset dimensions in quick mode. These are not part of the completed full-study comparison.

In [ ]:
other = runs[(runs.d != 5) & runs.guarded]
gen = across_seeds(other, ["d", "budget", "method"], "gap").rename(columns={"mean": "gap"}).merge(
    other.groupby(["d", "budget", "method"]).agg(value=("value", "mean"), median_test_se=("test_se", "median")).reset_index(),
    on=["d", "budget", "method"])
other_uncertainty = []
for (d, budget, method), sub in other.groupby(["d", "budget", "method"]):
    other_uncertainty.append(dict(d=d, budget=budget, method=method,
        **absolute_gap_uncertainty(sub.gap, sub.test_se)))
gen = gen.merge(pd.DataFrame(other_uncertainty), on=["d", "budget", "method"])
gen["reference"] = gen.d.map(lambda d: PAPER_REFERENCES[d]["value"])
gen["reference_ci"] = gen.d.map(lambda d: f"[{PAPER_REFERENCES[d]['low']}, {PAPER_REFERENCES[d]['high']}]")
gen["gap_inside_reference_band"] = [PAPER_REFERENCES[d]["value"] - PAPER_REFERENCES[d]["high"] <= g <= PAPER_REFERENCES[d]["value"] - PAPER_REFERENCES[d]["low"]
                                    for d, g in zip(gen.d, gen.gap)]
gen.to_csv(OUT / "summary_other_dimensions.csv", index=False)
gen.round(4)

## 3. Nested dual calculation

This cell estimates policy values and dual bounds with reduced outer and inner simulation budgets. It is a quick check of the estimator, not the source of the published primal–dual gaps.

The completed dual experiment evaluated the 50k, seed-zero LSM and neural policies on **2,000 outer paths with 3,200 inner continuations**, followed by a 6,400-inner sensitivity check on 500 outer paths. The reported paired gap is computed using an independently simulated initial policy-value estimate. Details are in [RESEARCH_NOTES.md](RESEARCH_NOTES.md).

In [ ]:
from dual import policy_dual

dual_outer, dual_inner = (32, 64) if MODE == "quick" else (2000, 3200)
dual_seeds = [s for s in (0, 1, 2) if s in study["seeds"]]
dual_plan = [(5, b, s, method, dual_inner)
             for b in study["budgets"] for s in dual_seeds
             for method in ("lsm", MAIN_VARIANT)]
dual_plan += [(5, b, 0, variant, dual_inner)
              for b in study["ablation_budgets"] for variant in VARIANTS if variant != MAIN_VARIANT]
dual_plan += [(2, study["case_2d"]["budget"], 0, method, dual_inner)
              for method in ("lsm", MAIN_VARIANT)]
dual_plan += [(5, max(study["budgets"]), 0, method, 2*dual_inner)
              for method in ("lsm", MAIN_VARIANT)]
dual_spec = dict(provenance=calibration_provenance(MODE), neural=frozen["neural"],
                 plan=dual_plan, outer=dual_outer, chunk=16, memory_mb=256,
                 training_validation=study, base_seed=BASE)
dual_folder = OUT / ("study-dual-" + stable_hash(dual_spec))
dual_folder.mkdir(exist_ok=True)
(dual_folder / "spec.json").write_text(json.dumps(dual_spec, indent=2))
dual_rows = []
for d, budget, seed, method, inner in dual_plan:
    stem = f"{d}d-{budget}-{seed}-{method}-{inner}"
    record_path = dual_folder / (stem + ".json")
    sample_path = dual_folder / (stem + ".npz")
    if record_path.exists() and sample_path.exists():
        dual_rows.append(json.loads(record_path.read_text()))
        continue
    o = cases[d]
    train = gbm_paths(o, budget, seed_for(BASE, "train", d, seed))
    val = gbm_paths(o, study["val"], seed_for(BASE, "val", d, seed))
    t0 = time.perf_counter()
    if method == "lsm":
        policy = select_lsm(o, train, val)[0]
    else:
        policy = fit_neural(o, train, val, with_variant(settings, method),
                            seed=seed_for(BASE, "init", d, seed))
    result = policy_dual(o, policy, outer=dual_outer, inner=inner,
                         base_seed=BASE, seed=86000+seed, outer_chunk=16, memory_mb=256)
    row = dict(d=d, budget=budget, seed=seed, method=method, inner=inner,
               seconds=time.perf_counter()-t0, metadata=result["metadata"],
               **{key: result[key] for key in ("lower", "upper", "gap")})
    np.savez_compressed(sample_path, lower=result["lower_samples"], upper=result["upper_samples"])
    temporary = record_path.with_suffix(".json.pending")
    temporary.write_text(json.dumps(row, indent=2, default=float))
    temporary.replace(record_path)
    dual_rows.append(row)
    print(stem, "gap:", row["gap"], flush=True)
dual_table = pd.DataFrame([dict(d=r["d"], budget=r["budget"], seed=r["seed"],
    method=r["method"], inner=r["inner"], **{f"{k}_{field}": r[k][field]
    for k in ("lower", "upper", "gap") for field in ("mean", "se", "ci_low", "ci_high")})
    for r in dual_rows])
dual_table.to_csv(dual_folder / "summary.csv", index=False)
display(dual_table)
stability = []
for method in ("lsm", MAIN_VARIANT):
    stem = f"5d-{max(study['budgets'])}-0-{method}-"
    with np.load(dual_folder / (stem+str(dual_inner)+".npz")) as a, np.load(
            dual_folder / (stem+str(2*dual_inner)+".npz")) as b:
        assert np.array_equal(a["lower"], b["lower"]), "outer paths differ"
        change = b["upper"]-a["upper"]
        mean, se, lo, hi = mean_ci(change)
        stability.append(dict(method=method, doubled_minus_base=mean, paired_se=se,
                              ci_low=lo, ci_high=hi))
pd.DataFrame(stability).to_csv(dual_folder / "inner_stability.csv", index=False)
display(pd.DataFrame(stability))


## 4. Completed study

The final data are in [`results/headline/H2.csv`](results/headline/H2.csv) and [`results/headline/H3.csv`](results/headline/H3.csv). The supporting [summary](results/supporting/summary.json) and [per-run records](results/supporting/runs.csv) contain further detail.

**Neural versus LSM.** Over ten training seeds per budget, the mean paired test-value differences (neural − LSM) were:

| Training paths | Difference | 95% interval across seeds |
|---:|---:|---:|
| 1,000 | −0.1233 | [−0.2482, 0.0017] |
| 10,000 | +0.0172 | [−0.0520, 0.0864] |
| 50,000 | +0.0036 | [−0.0353, 0.0425] |

At 1k, LSM was ahead in nine of ten seeds, but the prespecified t-test yielded **p = 0.053**. At 50k, there was no detectable difference. The across-seed interval is conditional on the shared 200k test paths; accounting for test-set sampling uncertainty makes equivalence within ±0.05 inconclusive.

**Primal–dual estimates.** For the two 50k, seed-zero policies:

| Policy | Estimated upper value | Estimated policy value | Gap (95% CI) |
|---|---:|---:|---:|
| LSM | 26.1559 | 26.0974 | 0.0586 [0.0443, 0.0728] |
| Neural | 26.1502 | 26.1039 | 0.0463 [0.0380, 0.0546] |

The gap is estimated with paired nested simulations. The doubled-inner sensitivity check showed only small changes, but its uncertainty does **not** rule out remaining inner-simulation bias. These upper estimates bound the optimal value **in expectation**, not path by path.

**Validation and limits.** The one-dimensional put agrees with CRR and convolution references at the reported numerical tolerance. The five-dimensional dual estimates are consistent with Becker et al.'s reference interval **[26.138, 26.174]**. Neither comparison proves implementation correctness or policy optimality. The proposed feature ablation, 2D extension and full neural tuning grid were not completed.

The original study is preserved at [`study-v1`](https://github.com/MoritzDue/bermudan-neural-stopping/tree/study-v1). A discrepancy between an archived execution hash and the current strict reproduction procedure remains disclosed; figures and saved summaries are available, but independent end-to-end rerunning is not established. See [RESEARCH_NOTES.md](RESEARCH_NOTES.md#reproducibility-and-scope).